# 01 Logistic Regression：用心脏病数据理解“预测概率”

这次不用抽象的 A 类、B 类。我们直接用 `heart_data.csv`：每一行代表一个人的一些身体指标，例如年龄、身高体重、血压、胆固醇等。

我们要让模型学习一个问题：**根据这些指标，预测 `cardio` 是 0 还是 1。**

- `cardio = 0`：数据里记录为没有心血管疾病
- `cardio = 1`：数据里记录为有心血管疾病

Logistic Regression 最重要的特点是：它不只是说“0 或 1”，还会给出“属于 1 的概率”。


## 运行前说明

这些 notebook 是教学演示，不是医学诊断或生物学结论。我们会使用你放在项目里的真实表格数据，但这里只是学习机器学习模型怎么工作。

数据路径：

- 心脏病数据：`../data/ml_data/heart_data.csv`
- 企鹅数据：`../data/ml_data/penguins.csv`

推荐在本项目已有的 Docker/Jupyter 环境里运行，因为 `environment/Dockerfile` 已经安装了 `numpy`、`pandas`、`scikit-learn`、`matplotlib`、`seaborn`。

如果你想在本机 Python 里运行，可以先安装：

```bash
pip install numpy pandas scikit-learn matplotlib seaborn notebook
```


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## 1. 读取并认识数据

先把 CSV 读进来，看前几行。机器学习的第一步通常不是训练模型，而是先搞清楚：每一列是什么意思、目标列是哪一列。


In [ ]:
data_path = Path('../data/ml_data/heart_data.csv')
heart = pd.read_csv(data_path)

print('数据形状：', heart.shape)
heart.head()


In [ ]:
print('cardio 标签分布：')
print(heart['cardio'].value_counts().sort_index())

plt.figure(figsize=(5, 4))
sns.countplot(data=heart, x='cardio', hue='cardio', palette=['#2b7bba', '#d95f02'], legend=False)
plt.title('Target label distribution: cardio')
plt.xlabel('cardio: 0=no record, 1=recorded disease')
plt.ylabel('Count')
plt.show()


## 2. 做一点点清理和改造

原始 `age` 是“天数”，对人来说不直观，所以我们把它换成年龄：`age_years`。

我们也会计算 BMI：

`BMI = 体重kg / 身高m²`

另外，真实数据里可能有输入错误，例如血压特别离谱。为了教学演示，我们只保留一个比较合理的范围。


In [ ]:
heart = heart.copy()
heart['age_years'] = heart['age'] / 365.25
heart['bmi'] = heart['weight'] / (heart['height'] / 100) ** 2

clean = heart.query(
    '120 <= height <= 220 and 35 <= weight <= 200 and 80 <= ap_hi <= 220 and 40 <= ap_lo <= 140 and 10 <= bmi <= 60'
).copy()

print('清理前行数：', len(heart))
print('清理后行数：', len(clean))
clean[['age_years', 'height', 'weight', 'bmi', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'cardio']].head()


## 3. 先用图感受规律

下面的图只画两个容易理解的指标：年龄和收缩压 `ap_hi`。

如果橙色点更多出现在右上方，就说明“年龄更大、血压更高”可能和 `cardio=1` 有关系。模型会同时看更多指标，不只看这两个。


In [ ]:
plot_sample = clean.sample(n=2500, random_state=RANDOM_STATE).copy()
plot_sample['cardio_label'] = plot_sample['cardio'].map({
    0: 'cardio=0: no record',
    1: 'cardio=1: recorded disease',
})

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=plot_sample,
    x='age_years',
    y='ap_hi',
    hue='cardio_label',
    hue_order=['cardio=0: no record', 'cardio=1: recorded disease'],
    palette={
        'cardio=0: no record': '#2b7bba',
        'cardio=1: recorded disease': '#d95f02',
    },
    alpha=0.45,
    s=35,
)
plt.title('Age and systolic blood pressure by cardio label')
plt.xlabel('Age in years')
plt.ylabel('Systolic blood pressure: ap_hi')
plt.legend(title='Meaning of point color')
plt.show()


## 4. 训练 Logistic Regression

我们选择一些容易理解的特征：

- `age_years`：年龄
- `bmi`：BMI
- `ap_hi`、`ap_lo`：血压
- `cholesterol`、`gluc`：胆固醇、血糖等级
- `smoke`、`alco`、`active`：是否吸烟、饮酒、运动

Logistic Regression 会学到每个指标的“方向和重要程度”，再输出一个 0 到 1 之间的概率。


In [ ]:
# 这些列是模型可以看的输入信息，也叫 features（特征）。
features = ['age_years', 'bmi', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

# 这一列是模型要学习预测的答案，也叫 target（目标）。
target = 'cardio'

# 为了 notebook 运行更快，抽样 12000 行；数据依然足够演示。
model_data = clean[features + [target]].sample(n=12000, random_state=RANDOM_STATE)

# X 只保存输入特征：年龄、BMI、血压等。
X = model_data[features]

# y 只保存正确答案：cardio 是 0 还是 1。
y = model_data[target]

# train_test_split 会把数据分成训练集和测试集。
X_train, X_test, y_train, y_test = train_test_split(
    # X 是所有输入特征，y 是对应的正确答案。
    X, y,
    # test_size=0.25 表示 25% 的数据留作测试，75% 的数据用于训练。
    test_size=0.25,
    # random_state 固定随机种子，保证每次运行切分结果一样，方便复现。
    random_state=RANDOM_STATE,
    # stratify=y 保证训练集和测试集里的 cardio=0/1 比例尽量和原数据一致。
    stratify=y
)

# make_pipeline 把多个步骤串起来：先标准化，再训练逻辑回归模型。
model = make_pipeline(
    # StandardScaler 把不同单位的数字拉到相近尺度，例如年龄、血压、BMI 不会互相压过对方。
    StandardScaler(),
    # LogisticRegression 是真正的预测模型；max_iter=1000 给它足够次数去学习，random_state 用于复现。
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
)

# fit 表示“开始学习”：模型用训练集 X_train 和答案 y_train 找规律。
model.fit(X_train, y_train)

print('训练完成')
print('测试集准确率：', round(model.score(X_test, y_test), 3))


## 5. 预测一个具体的人

下面这个人 **不是从原始样本里复制出来的**，而是我们手工写的一个假想例子，只是为了演示模型怎么使用。

模型会输出：

- 预测类别：0 或 1
- 预测概率：更像 0 还是更像 1

这里要区分两个数字：

- **测试集准确率**：模型在很多测试样本上总体猜对了多少，例如 `0.74` 表示大约 74% 猜对。
- **单个人的预测概率**：模型对某一个输入样本有多偏向 `cardio=1` 或 `cardio=0`。

所以 `0.74` 如果出现在“测试集准确率”那里，它不是某个人的风险概率，而是整个模型的考试成绩。


In [ ]:
# 这是一个手工创建的假想人物，不是直接从 CSV 样本里拿出来的一行。
new_person = pd.DataFrame({
    'age_years': [55],
    'bmi': [28],
    'ap_hi': [145],
    'ap_lo': [90],
    'cholesterol': [2],
    'gluc': [1],
    'smoke': [0],
    'alco': [0],
    'active': [1],
})

predicted_label = model.predict(new_person)[0]
predicted_proba = model.predict_proba(new_person)[0]

same_as_existing_row = (clean[features].round(6) == new_person.iloc[0].round(6)).all(axis=1).any()

risk_probability = predicted_proba[1]
if risk_probability >= 0.80:
    probability_level = 'high'
elif risk_probability >= 0.60:
    probability_level = 'medium-high'
elif risk_probability >= 0.40:
    probability_level = 'uncertain / near the middle'
else:
    probability_level = 'low'

display(new_person)
print('Is this exact person copied from the CSV data?', same_as_existing_row)
print('Predicted cardio:', predicted_label)
print('Probability for cardio=0:', round(predicted_proba[0], 3))
print('Probability for cardio=1:', round(predicted_proba[1], 3))
print('How to read the cardio=1 probability:', probability_level)


## 6. 评估：模型到底猜得怎么样？

准确率表示“测试集中有多少比例预测对了”。如果测试集准确率大约是 `0.74`，意思是：100 个测试样本里，模型大约猜对 74 个。

这个成绩怎么判断？

- 对入门教学来说，`0.74` 说明模型确实学到了一些规律，比乱猜好。
- 对这份数据来说，`cardio=0` 和 `cardio=1` 数量差不多，乱猜大概接近 `0.50`，所以 `0.74` 明显高于乱猜。
- 但对健康/医学类预测来说，`0.74` 不算很高，不能拿来做严肃诊断，只适合学习模型流程。

如果想提高测试集准确率，重点不是“把某一次预测概率调高”，而是让模型真的学得更好。常见方法包括：清理异常血压/身高体重数据、加入更有用的特征、尝试 Random Forest 等非线性模型、调节模型参数，并且始终用测试集或交叉验证确认有没有真的变好。

混淆矩阵能看得更细：哪些 0 被猜成 1，哪些 1 被猜成 0。


In [ ]:
y_pred = model.predict(X_test)

print('准确率：', round(accuracy_score(y_test, y_pred), 3))
print()
print('分类报告：')
print(classification_report(y_test, y_pred, target_names=['cardio=0', 'cardio=1']))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['cardio=0', 'cardio=1'])
disp.plot(cmap='Blues')
plt.title('Logistic Regression Confusion Matrix')
plt.show()


## 7. 如何提高测试集准确率：调参数

调参数的意思是：模型种类不变，还是 Logistic Regression，但我们试几种不同设置，看看哪一种在验证中表现最好。

这里先调 3 个常见参数：

- `C`：控制模型有多“大胆”。`C` 小一点，模型更保守；`C` 大一点，模型更努力贴合训练数据。
- `class_weight`：如果某一类更难猜，可以让模型更重视类别平衡。这里数据大致平衡，但仍可以试试。
- `solver`：求解方法，可以理解成模型找答案时用的计算方法。

重要提醒：调参不是保证准确率一定上升。它只是更系统地试设置。最后一定要看测试集表现，而不是只看训练集。


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# 先记录原始模型在测试集上的准确率，方便后面比较。
baseline_accuracy = model.score(X_test, y_test)

# 这个 pipeline 和前面一样：先标准化，再训练 Logistic Regression。
tuning_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
)

# param_grid 写出我们想尝试的参数组合。
# 名字前面的 logisticregression__ 表示：这个参数属于 pipeline 里的 LogisticRegression 那一步。
param_grid = {
    'logisticregression__C': [0.01, 0.1, 1, 10],
    'logisticregression__class_weight': [None, 'balanced'],
    'logisticregression__solver': ['lbfgs', 'liblinear'],
}

# 交叉验证：把训练集再分成 5 份，多轮验证，避免只靠一次切分就下结论。
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# GridSearchCV 会尝试 param_grid 里的所有组合，选择平均验证准确率最高的一组。
grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
tuned_accuracy = best_model.score(X_test, y_test)

print('Baseline test accuracy:', round(baseline_accuracy, 3))
print('Best CV accuracy during tuning:', round(grid_search.best_score_, 3))
print('Tuned test accuracy:', round(tuned_accuracy, 3))
print('Best parameters:')
print(grid_search.best_params_)


### 怎么读调参结果？

- 如果 `Tuned test accuracy` 比 `Baseline test accuracy` 高，说明这次调参在测试集上有帮助。
- 如果差不多，说明原来的默认参数已经够用了。
- 如果反而更低，说明调参可能只是让模型在交叉验证里看起来更好，但没有真正提升测试集表现。

对 Logistic Regression 来说，调参数能带来的提升通常有限。想明显提高准确率，常见方向还有：

- 加入更有信息量的特征，例如脉压 `ap_hi - ap_lo`、年龄分组、BMI 分组。
- 更认真地处理异常值，例如不合理的血压或身高体重。
- 尝试能学习非线性关系的模型，例如 Random Forest。
- 用交叉验证比较多个模型，而不是只看一次 train/test split。


## 8. 如何提高测试集准确率：特征工程

特征工程的意思是：不只是把原始列直接丢给模型，而是根据常识和数据含义，创造一些更容易被模型理解的新列。

比如原始数据里有：

- `ap_hi`：收缩压
- `ap_lo`：舒张压

我们可以创造：

- `pulse_pressure = ap_hi - ap_lo`：脉压
- `mean_arterial_pressure = (ap_hi + 2 * ap_lo) / 3`：平均动脉压的近似值

再比如模型可能不容易自己理解“年龄和血压一起变高”这件事，我们可以加交互特征：

- `age_ap_hi = age_years * ap_hi`
- `age_bmi = age_years * bmi`

这些新列不一定都有效，所以最后还是要看测试集准确率。


In [ ]:
from sklearn.metrics import roc_auc_score

# 从前面同一份 model_data 开始，保证比较公平：样本完全一样。
engineered_data = model_data.copy()

# 血压相关的新特征。
engineered_data['pulse_pressure'] = engineered_data['ap_hi'] - engineered_data['ap_lo']
engineered_data['mean_arterial_pressure'] = (engineered_data['ap_hi'] + 2 * engineered_data['ap_lo']) / 3

# 交互特征：让线性模型也能看到“两个指标一起变化”的效果。
engineered_data['age_bmi'] = engineered_data['age_years'] * engineered_data['bmi']
engineered_data['age_ap_hi'] = engineered_data['age_years'] * engineered_data['ap_hi']
engineered_data['bmi_ap_hi'] = engineered_data['bmi'] * engineered_data['ap_hi']

# 阈值特征：把一些容易理解的判断变成 0/1。
engineered_data['high_cholesterol'] = (engineered_data['cholesterol'] >= 2).astype(int)
engineered_data['very_high_cholesterol'] = (engineered_data['cholesterol'] >= 3).astype(int)
engineered_data['high_gluc'] = (engineered_data['gluc'] >= 2).astype(int)
engineered_data['stage2_systolic'] = (engineered_data['ap_hi'] >= 140).astype(int)
engineered_data['stage2_diastolic'] = (engineered_data['ap_lo'] >= 90).astype(int)
engineered_data['older_than_50'] = (engineered_data['age_years'] >= 50).astype(int)
engineered_data['older_than_55'] = (engineered_data['age_years'] >= 55).astype(int)

engineered_features = features + [
    'pulse_pressure',
    'mean_arterial_pressure',
    'age_bmi',
    'age_ap_hi',
    'bmi_ap_hi',
    'high_cholesterol',
    'very_high_cholesterol',
    'high_gluc',
    'stage2_systolic',
    'stage2_diastolic',
    'older_than_50',
    'older_than_55',
]

X_eng = engineered_data[engineered_features]
y_eng = engineered_data[target]

X_eng_train, X_eng_test, y_eng_train, y_eng_test = train_test_split(
    X_eng, y_eng, test_size=0.25, random_state=RANDOM_STATE, stratify=y_eng
)

feature_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)
)

feature_model.fit(X_eng_train, y_eng_train)

baseline_proba = model.predict_proba(X_test)[:, 1]
feature_proba = feature_model.predict_proba(X_eng_test)[:, 1]

baseline_accuracy = model.score(X_test, y_test)
feature_accuracy = feature_model.score(X_eng_test, y_eng_test)

comparison = pd.DataFrame({
    'model': ['baseline logistic regression', 'with feature engineering'],
    'test_accuracy': [baseline_accuracy, feature_accuracy],
    'roc_auc': [roc_auc_score(y_test, baseline_proba), roc_auc_score(y_eng_test, feature_proba)],
})

comparison


### 特征工程 + 调参一起用

刚才是“只加新特征”。现在我们把新特征和前面的 `GridSearchCV` 合起来：让模型既看到更好的输入，也自动尝试几组参数。


In [ ]:
engineered_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)
)

engineered_param_grid = {
    'logisticregression__C': [0.01, 0.03, 0.1, 0.3, 1, 3, 10],
    'logisticregression__class_weight': [None, 'balanced'],
    'logisticregression__solver': ['lbfgs', 'liblinear'],
}

engineered_grid_search = GridSearchCV(
    estimator=engineered_pipeline,
    param_grid=engineered_param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
)

engineered_grid_search.fit(X_eng_train, y_eng_train)

best_engineered_model = engineered_grid_search.best_estimator_
engineered_tuned_accuracy = best_engineered_model.score(X_eng_test, y_eng_test)
engineered_tuned_proba = best_engineered_model.predict_proba(X_eng_test)[:, 1]
engineered_tuned_auc = roc_auc_score(y_eng_test, engineered_tuned_proba)

final_comparison = pd.DataFrame({
    'model': [
        'baseline logistic regression',
        'tuned baseline logistic regression',
        'feature engineering only',
        'feature engineering + tuning',
    ],
    'test_accuracy': [
        baseline_accuracy,
        tuned_accuracy,
        feature_accuracy,
        engineered_tuned_accuracy,
    ],
    'roc_auc': [
        roc_auc_score(y_test, baseline_proba),
        roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_eng_test, feature_proba),
        engineered_tuned_auc,
    ],
})

print('Best parameters after feature engineering:')
print(engineered_grid_search.best_params_)

display(final_comparison)


### 这次提升说明什么？

在这份数据和这次切分下，特征工程通常会带来一点提升，但不会让 Logistic Regression 一下子从 0.74 变成 0.90。

原因是：Logistic Regression 本质上还是线性模型。新特征可以帮它看到更多规律，但如果真实规律很复杂，可能需要 Random Forest 这种更会处理非线性关系的模型。

所以更现实的学习结论是：

- 调参能微调模型。
- 特征工程通常比只调参数更有用。
- 如果提升仍然很小，说明可能要换模型，或者需要更有信息量的数据。


## 9. 看每个特征的方向

Logistic Regression 会给每个特征一个系数。简单理解：

- 系数为正：这个特征变大时，更容易预测为 `cardio=1`
- 系数为负：这个特征变大时，更容易预测为 `cardio=0`

注意：这只是模型在这份数据里学到的统计关系，不等于医学因果结论。


In [ ]:
log_reg = model.named_steps['logisticregression']
coef = pd.Series(log_reg.coef_[0], index=features).sort_values()

plt.figure(figsize=(7, 5))
sns.barplot(x=coef.values, y=coef.index, hue=coef.index, palette='vlag', legend=False)
plt.axvline(0, color='black', linewidth=1)
plt.title('Logistic Regression Feature Coefficients')
plt.xlabel('Coefficient: positive leans toward cardio=1, negative leans toward cardio=0')
plt.ylabel('Feature')
plt.show()

coef


## 10. 小结

Logistic Regression 适合用在：

- 想知道“某类的概率”时。
- 想要一个相对容易解释的模型时。
- 数据规律比较接近“指标加起来形成一个风险分数”时。

在这个例子里，你可以把它想成：模型根据年龄、BMI、血压等信息，算出一个倾向分数，再把分数变成 `cardio=1` 的概率。
